# Metadata Enrichment Pipeline v2 — Generic, Retrieval-Augmented

**Architecture changes from v1:**

1. **Chunking**: Supports both **flat** and **parent-child** chunking
2. **Extraction**: **Retrieval-augmented** (primary) — per-field retrieve → extract → label; **section-based** (fallback)
3. **Context window**: Never exceeds limit — only retrieved chunks or sections sent to LLM
4. **Per-chunk metadata**: Retrieved chunks get extracted value; others get **explicit null**
5. **Generic/config-driven**: Schema-agnostic, works for any document type

## 1. Setup and Configuration

In [ ]:
!pip install -q google-genai elasticsearch sentence-transformers pydantic tqdm numpy docling

In [1]:
import os
import hashlib
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Any, Type, Literal

from pydantic import BaseModel, Field, create_model
from google import genai
from elasticsearch import Elasticsearch, helpers
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np

In [2]:
# # ── Gemini LLM config ─────────────────────────────────────────────────────────
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")
# GEMINI_MODEL = "gemini-2.0-flash"

# ── Elasticsearch config ──────────────────────────────────────────────────────
ES_HOST = "http://localhost:9200"
ES_INDEX = "lease_documents"
ES_USER = "elastic"
ES_PASSWORD = "changeme"

# ── Embedding config ──────────────────────────────────────────────────────────
EMBEDDING_MODEL_NAME = "Snowflake/snowflake-arctic-embed-m"
EMBEDDING_DIM = 768

# ── Chunking config ──────────────────────────────────────────────────────────
PARENT_CHUNK_SIZE = 2000
PARENT_CHUNK_OVERLAP = 200
CHILD_CHUNK_SIZE = 512
CHILD_CHUNK_OVERLAP = 50

# ── Data sources ──────────────────────────────────────────────────────────────
DATA_DIR = Path("./data/leases").resolve()
SAMPLE_PDF_URLS: List[str] = [
    "https://esign.com/wp-content/uploads/Standard-Residential-Lease-Agreement-Template.pdf",
    "https://rentalleaseagreements.com/wp-content/uploads/2020/07/Standard-Residential-Lease-Agreement.pdf",
    "https://rentallease.com/wp-content/uploads/california-standard-residential-lease-agreement-form.pdf",
    "https://freeforms.com/wp-content/uploads/2021/01/Standard-Lease-Agreement-Template.pdf",
    "https://www.dottedsign.com/blog/wp-content/uploads/2024/08/California-Standard-Residential-Lease-Agreement.pdf",
    "https://esign.com/wp-content/uploads/Texas-Standard-Residential-Lease-Agreement.pdf",
    "https://esign.com/wp-content/uploads/New-York-Standard-Residential-Lease-Agreement.pdf",
    "https://eforms.com/images/2015/10/California-Month-to-Month-Rental-Agreement-Template.pdf",
    "https://ipropertymanagement.com/wp-content/uploads/6891/Standard-California-Residential-Lease-Agreement-Template.pdf",
    "https://esign.com/wp-content/uploads/Sublease-Agreement-Template.pdf",
    "https://www.freeprintablelegalforms.com/downloads/2024/04/Sublease-Agreement-Form.pdf",
]

In [3]:
@dataclass
class EnrichmentConfig:
    """Generic config — works for any document type."""
    metadata_schema: Type[BaseModel]
    field_retrieval_queries: Dict[str, List[str]] = None  # deprecated; use json_schema_extra keywords on schema fields
    extraction_strategy: Literal["retrieval_augmented", "section_based", "hybrid"] = "retrieval_augmented"
    context_limit_tokens: int = 8000
    chars_per_token: float = 4.0
    safety_margin: float = 0.7
    chunks_per_section: Optional[int] = None
    merge_strategy: Literal["first_non_null", "last_non_null", "most_common"] = "first_non_null"
    chunking_mode: Literal["flat", "parent_child"] = "flat"
    top_k_per_field: int = 5

In [4]:
# gemini_client = genai.Client(api_key=GEMINI_API_KEY)

es = Elasticsearch(
    hosts=[ES_HOST],
    basic_auth=(ES_USER, ES_PASSWORD) if ES_USER and ES_PASSWORD else None,
    verify_certs=False,
)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)

print("ES:", es.info().body.get("version", {}).get("number", "?"))
# print("Gemini model:", GEMINI_MODEL)
print("Embedding model:", EMBEDDING_MODEL_NAME, "(dim=%d)" % EMBEDDING_DIM)

ES: 8.12.0
Embedding model: Snowflake/snowflake-arctic-embed-m (dim=768)


## 2. Sample Data and PDF Ingestion

Place your lease/amendment PDFs in `./data/leases/` (create the folder if needed). Optionally set `SAMPLE_PDF_URLS` in config to download PDFs from URLs. Docling will extract text from each PDF.

In [5]:
# Ensure data directory exists; fetch sources for sample PDFs, download and save to DATA_DIR
DATA_DIR.mkdir(parents=True, exist_ok=True)

def download_pdf(url: str, dest: Path) -> bool:
    """Download PDF from URL and save to dest. Uses User-Agent to avoid blocks. Returns True on success."""
    import urllib.request
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; rv:91.0) Gecko/20100101 Firefox/91.0"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            dest.write_bytes(resp.read())
        return True
    except Exception as e:
        print("Download failed %s: %s" % (url, e))
        return False

def get_pdf_paths() -> List[Path]:
    """Collect PDF paths: local folder first, then download from SAMPLE_PDF_URLS and save to DATA_DIR."""
    paths: List[Path] = []
    for f in DATA_DIR.glob("*.pdf"):
        paths.append(f.resolve())
    for i, url in enumerate(SAMPLE_PDF_URLS):
        try:
            name = url.rstrip("/").split("/")[-1] or "downloaded.pdf"
            if not name.lower().endswith(".pdf"):
                name = "downloaded_%d.pdf" % i
            dest = DATA_DIR / name
            if not dest.exists():
                if download_pdf(url, dest):
                    print("Downloaded: %s -> %s" % (url, dest))
            if dest.exists():
                paths.append(dest.resolve())
            else:
                print("Skipped (file missing): %s" % dest.name)
        except Exception as e:
            print("Skip URL %s: %s" % (url, e))
    return sorted(set(paths))

pdf_paths = get_pdf_paths()
print("PDFs to process:", len(pdf_paths))
for p in pdf_paths[:10]:
    print("  ", p)
if not pdf_paths:
    print("Add PDFs to %s or set SAMPLE_PDF_URLS to run the pipeline." % DATA_DIR)

Download failed https://eforms.com/images/2015/10/California-Month-to-Month-Rental-Agreement-Template.pdf: HTTP Error 403: Forbidden
Skipped (file missing): California-Month-to-Month-Rental-Agreement-Template.pdf
Download failed https://ipropertymanagement.com/wp-content/uploads/6891/Standard-California-Residential-Lease-Agreement-Template.pdf: HTTP Error 403: Forbidden
Skipped (file missing): Standard-California-Residential-Lease-Agreement-Template.pdf
PDFs to process: 9
   C:\Users\hp\OneDrive\Desktop\Personal_Projects\RAG_METADATA\data\leases\california-standard-residential-lease-agreement-form.pdf
   C:\Users\hp\OneDrive\Desktop\Personal_Projects\RAG_METADATA\data\leases\California-Standard-Residential-Lease-Agreement.pdf
   C:\Users\hp\OneDrive\Desktop\Personal_Projects\RAG_METADATA\data\leases\New-York-Standard-Residential-Lease-Agreement.pdf
   C:\Users\hp\OneDrive\Desktop\Personal_Projects\RAG_METADATA\data\leases\Standard-Lease-Agreement-Template.pdf
   C:\Users\hp\OneDrive\De

### PDF ingestion with Docling

Docling converts PDFs (and other formats) to structured text. We export to markdown and use that as `full_text` for chunking. If Docling fails (e.g. **std::bad_alloc** / out-of-memory on large or image-heavy pages), we **fall back to PyMuPDF** so the pipeline still gets text and continues.

In [6]:
from docling.document_converter import DocumentConverter

def _load_pdf_with_pymupdf(path: Path) -> str:
    """Fallback: extract text with PyMuPDF when Docling fails (e.g. std::bad_alloc / OOM)."""
    try:
        import fitz
        doc = fitz.open(path)
        text = "".join(page.get_text() for page in doc)
        doc.close()
        return text or ""
    except Exception:
        return ""

def load_pdf_with_docling(path: Path, use_fallback_on_error: bool = True) -> Dict[str, Any]:
    """Convert a single PDF to text using Docling. On failure (e.g. bad_alloc/OOM), fall back to PyMuPDF."""
    try:
        converter = DocumentConverter()
        result = converter.convert(str(path))
        doc = result.document
        full_text = doc.export_to_markdown() if hasattr(doc, "export_to_markdown") else str(doc)
        if not full_text or not full_text.strip():
            full_text = ""
    except Exception as e:
        full_text = ""
        if use_fallback_on_error:
            full_text = _load_pdf_with_pymupdf(path)
            if not full_text:
                print("  [Fallback] %s: Docling failed (%s), PyMuPDF returned no text." % (path.name, e))
            else:
                print("  [Fallback] %s: Docling failed (%s), used PyMuPDF (%d chars)." % (path.name, e, len(full_text)))
        else:
            raise
    try:
        mtime = path.stat().st_mtime
        upload_date = datetime.fromtimestamp(mtime).strftime("%Y-%m-%d")
    except Exception:
        upload_date = datetime.utcnow().strftime("%Y-%m-%d")
    title = path.stem.replace("_", " ").replace("-", " ")
    return {
        "file_name": path.name,
        "title": title,
        "upload_date": upload_date,
        "full_text": full_text if full_text else "",
    }

# Load all PDFs (Docling; on std::bad_alloc / OOM we fall back to PyMuPDF)
raw_documents: List[Dict[str, Any]] = []
for path in tqdm(pdf_paths, desc="Docling PDF"):
    try:
        raw_documents.append(load_pdf_with_docling(path))
    except Exception as e:
        print("Skip %s: %s" % (path, e))

print("Loaded %d documents." % len(raw_documents))
for d in raw_documents[:3]:
    print("  %s: %d chars" % (d["file_name"], len(d.get("full_text", ""))))

Docling PDF:   0%|          | 0/9 [00:00<?, ?it/s]

[INFO] 2026-03-18 12:39:10,344 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-18 12:39:10,377 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-18 12:39:10,441 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\hp\OneDrive\Desktop\Personal_Projects\RAG_METADATA\agentic_rag\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-18 12:39:10,441 [RapidOCR] main.py:50: Using C:\Users\hp\OneDrive\Desktop\Personal_Projects\RAG_METADATA\agentic_rag\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-18 12:39:10,838 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-18 12:39:10,838 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-18 12:39:10,855 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\hp\OneDrive\Desktop\Personal_Projects\RAG_METADATA\agentic_rag\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-18 12:39:10,859 [RapidOC

Loaded 9 documents.
  california-standard-residential-lease-agreement-form.pdf: 26715 chars
  California-Standard-Residential-Lease-Agreement.pdf: 20817 chars
  New-York-Standard-Residential-Lease-Agreement.pdf: 20580 chars






























































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































































### Parent-child chunking

Parents (~2000 chars) preserve context for extraction; children (~512 chars) are embedded for precise retrieval. Each child has `parent_id`. **Initial metadata** is a **generic filter**: keys are derived from the metadata schema (when available), so they are not hardcoded; **values are not available upfront** — they are all `None` here. They get **populated during the enrichment step** when we run extraction (retrieval-augmented or section-based) and the LLM fills `chunk["metadata"]` from the document. If no schema is set yet, a minimal generic set (`document_type`, `source_file`) is used.

In [7]:
# Initial metadata: generic filter — keys are not known upfront; they are derived from the
# metadata schema (extraction target). Use schema when available so chunks match index mapping.
GENERIC_METADATA_KEYS = ["document_type", "source_file"]  # fallback when no schema provided

def get_initial_metadata(metadata_schema: Optional[Type[BaseModel]] = None) -> Dict[str, Any]:
    """Build initial metadata dict (all None). Schema-driven when provided; else generic keys."""
    if metadata_schema is not None:
        return {k: None for k in metadata_schema.model_fields.keys()}
    return {k: None for k in GENERIC_METADATA_KEYS}

def deterministic_id(content: str, prefix: str = "") -> str:
    h = hashlib.sha256(content.encode("utf-8")).hexdigest()[:16]
    return (prefix + h) if prefix else h

def split_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def create_parent_child_chunks(document: Dict[str, Any], metadata_schema: Optional[Type[BaseModel]] = None) -> Dict[str, Any]:
    """Produce parent and child chunks; each chunk gets initial metadata (all None). Keys from schema or generic."""
    full_text = document["full_text"] or ""
    file_name = document["file_name"]
    doc_id = deterministic_id(file_name, prefix="doc_")
    initial_metadata = get_initial_metadata(metadata_schema)

    parent_texts = split_text(full_text, PARENT_CHUNK_SIZE, PARENT_CHUNK_OVERLAP)
    parent_chunks = []
    child_chunks = []

    for p_idx, p_text in enumerate(parent_texts):
        parent_id = deterministic_id("%s_%d" % (doc_id, p_idx), prefix="parent_")
        parent_chunks.append({
            "chunk_id": parent_id,
            "doc_id": doc_id,
            "chunk_type": "parent",
            "chunk_index": p_idx,
            "text": p_text,
            "file_name": file_name,
            "title": document.get("title", file_name),
            "upload_date": document.get("upload_date", ""),
            "metadata": dict(initial_metadata),
        })

        child_texts = split_text(p_text, CHILD_CHUNK_SIZE, CHILD_CHUNK_OVERLAP)
        for c_idx, c_text in enumerate(child_texts):
            child_id = deterministic_id("%s_%d" % (parent_id, c_idx), prefix="child_")
            child_chunks.append({
                "chunk_id": child_id,
                "parent_id": parent_id,
                "doc_id": doc_id,
                "chunk_type": "child",
                "chunk_index": c_idx,
                "text": c_text,
                "file_name": file_name,
                "title": document.get("title", file_name),
                "upload_date": document.get("upload_date", ""),
                "metadata": dict(initial_metadata),
            })

    document["doc_id"] = doc_id
    document["parent_chunks"] = parent_chunks
    document["child_chunks"] = child_chunks
    return document

if raw_documents:
    try:
        _schema = config.metadata_schema
    except NameError:
        _schema = None
    for doc in raw_documents:
        create_parent_child_chunks(doc, metadata_schema=_schema)
    total_p = sum(len(d["parent_chunks"]) for d in raw_documents)
    total_c = sum(len(d["child_chunks"]) for d in raw_documents)
    print("Parent-child chunking: %d parents, %d children" % (total_p, total_c))

Parent-child chunking: 94 parents, 449 children


### Embeddings (Snowflake Arctic Embed M)

Only **child** chunks are embedded for vector search. Parents are used for context at retrieval and for metadata extraction. Uses **batching** (configurable `batch_size`) and optional **multi-process parallelization** (`n_workers` > 1) to speed up encoding.

In [8]:
def embed_child_chunks(
    documents: List[Dict[str, Any]],
    model: SentenceTransformer,
    batch_size: int = 64,
) -> None:
    """Encode child chunk texts and attach embeddings in place."""
    all_children = [c for doc in documents for c in doc.get("child_chunks", [])]
    if not all_children:
        return
    texts = [c["text"] for c in all_children]
    embs = model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
    for child, emb in zip(all_children, embs):
        child["embedding"] = emb.tolist()

if raw_documents:
    embed_child_chunks(raw_documents, embedding_model)
    print("Embeddings attached to %d child chunks." % sum(len(d.get("child_chunks", [])) for d in raw_documents))

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embeddings attached to 449 child chunks.


## 3. Metadata Schema and Field Retrieval Queries

Once chunks are **embedded and stored in Elasticsearch with basic indexing**, we **enrich metadata using the LLM**. We define the metadata fields here (schema) and the queries used to **pull relevant chunks** where each field can be found. Enrichment is done **first at document level** (one metadata per `doc_id`), then we **re-index** (bulk-update ES) with the enriched metadata.

In [9]:
# Each field has:
#   description  → used for (1) retrieval as the primary semantic query, and (2) the LLM prompt.
#                  Write it as a clear, natural sentence (no "Include:", "Look for:" boilerplate).
#   keywords     → optional list in json_schema_extra; extra short queries for BM25 recall.
#                  Retrieval runs: [description] + keywords, fused via RRF.
# Everything lives in one place per field — no separate FIELD_RETRIEVAL_QUERIES dict to maintain.

class LeaseMetadata(BaseModel):
    owner_name: Optional[str] = Field(
        None,
        description="Name of the tenant, lessee, or occupant who rents the property",
        json_schema_extra={"keywords": [
            "tenant name", "lessee", "party A", "occupant", "resident",
            "renter", "leaseholder", "named tenant",
        ]},
    )
    landlord: Optional[str] = Field(
        None,
        description="Name of the landlord, lessor, or property owner who leases out the property",
        json_schema_extra={"keywords": [
            "landlord", "lessor", "property owner", "party B",
            "management company", "property manager",
        ]},
    )
    signing_date: Optional[str] = Field(
        None,
        description="Date the lease was signed or executed, in YYYY-MM-DD format",
        json_schema_extra={"keywords": [
            "signed on", "execution date", "lease date", "effective date",
            "commencement date", "date of execution", "agreement date",
        ]},
    )
    expiry_date: Optional[str] = Field(
        None,
        description="Date the lease expires or terminates, in YYYY-MM-DD format",
        json_schema_extra={"keywords": [
            "expiration", "lease term ends", "termination date", "end date",
            "lease expires", "term end", "conclusion of term",
        ]},
    )
    escalation_clause: Optional[str] = Field(
        None,
        description="Rent escalation or adjustment clause describing how rent changes over time",
        json_schema_extra={"keywords": [
            "rent escalation", "annual increase", "rent adjustment",
            "percentage increase", "CPI", "cost-of-living", "rent increase",
        ]},
    )
    property_address: Optional[str] = Field(
        None,
        description="Full address of the leased property or premises",
        json_schema_extra={"keywords": [
            "property address", "premises", "located at", "street address",
            "leased property", "subject property", "rental unit address",
        ]},
    )
    monthly_rent: Optional[str] = Field(
        None,
        description="Monthly rent or base rent amount including currency",
        json_schema_extra={"keywords": [
            "monthly rent", "base rent", "rent amount", "rental amount",
            "rent due", "lease payment",
        ]},
    )
    amendment_number: Optional[int] = Field(
        None,
        description="Amendment number (1, 2, etc.) or None for the original lease",
        json_schema_extra={"keywords": [
            "amendment", "first amendment", "second amendment",
            "amendment no", "addendum", "modification",
        ]},
    )
    document_type: Optional[str] = Field(
        None,
        description="Document type such as lease, amendment, addendum, or renewal",
        json_schema_extra={"keywords": [
            "lease agreement", "amendment", "lease", "addendum",
            "residential lease", "standard lease", "modification agreement",
        ]},
    )

# Flow: for each field, retrieval runs [description] + keywords → LLM extracts value → document-level metadata → re-index ES

In [10]:
config = EnrichmentConfig(
    metadata_schema=LeaseMetadata,
    extraction_strategy="retrieval_augmented",
    chunking_mode="parent_child",
    top_k_per_field=5,
)

### Create Elasticsearch index and index documents (basic indexing)

Index mapping: `dense_vector` (768 dims) for KNN, `parent_id` for parent-child, and `metadata` object with schema-derived fields. Bulk-index parents (no embedding) then children (with embedding); both carry the metadata layer (initial values are None). **Enrichment at document level and re-index (update) happen later** in Section 9.

In [11]:
def build_metadata_mapping(schema: Type[BaseModel]) -> Dict[str, Any]:
    """Derive ES field mappings from Pydantic metadata schema (for metadata.*)."""
    type_map = {"string": {"type": "keyword"}, "integer": {"type": "integer"}, "number": {"type": "float"}, "boolean": {"type": "boolean"}}
    json_schema = schema.model_json_schema()
    properties = {}
    for field_name, field_def in json_schema.get("properties", {}).items():
        if "anyOf" in field_def:
            for option in field_def["anyOf"]:
                if option.get("type") != "null":
                    properties[field_name] = type_map.get(option["type"], {"type": "keyword"})
                    break
        else:
            properties[field_name] = type_map.get(field_def.get("type", "string"), {"type": "keyword"})
    return properties

def create_es_index(index_name: str, schema: Type[BaseModel], embedding_dim: int):
    """Create ES index for parent+child chunks with dense_vector and metadata."""
    metadata_fields = build_metadata_mapping(schema)
    mapping = {
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {
            "properties": {
                "chunk_id": {"type": "keyword"},
                "parent_id": {"type": "keyword"},
                "doc_id": {"type": "keyword"},
                "chunk_type": {"type": "keyword"},
                "chunk_index": {"type": "integer"},
                "text": {"type": "text", "analyzer": "standard"},
                "file_name": {"type": "keyword"},
                "title": {"type": "text", "fields": {"raw": {"type": "keyword"}}},
                "upload_date": {"type": "date"},
                "embedding": {"type": "dense_vector", "dims": embedding_dim, "index": True, "similarity": "cosine"},
                "metadata": {"type": "object", "properties": metadata_fields},
            }
        },
    }
    if es.indices.exists(index=index_name):
        es.indices.delete(index=index_name)
    es.indices.create(index=index_name, body=mapping)
    print("Index '%s' created (dims=%d, metadata fields: %s)" % (index_name, embedding_dim, list(metadata_fields.keys())))
    print("  Index structure: chunk_id, doc_id, chunk_type, text, file_name, title, upload_date, metadata (object);")
    print("  Parents: no 'embedding', no 'parent_id'. Children: have 'embedding' (dense_vector) + 'parent_id' for KNN + join.")

create_es_index(ES_INDEX, LeaseMetadata, EMBEDDING_DIM)
print("Index name:", ES_INDEX)

Index 'lease_documents' created (dims=768, metadata fields: ['owner_name', 'landlord', 'signing_date', 'expiry_date', 'escalation_clause', 'property_address', 'monthly_rent', 'amendment_number', 'document_type'])
  Index structure: chunk_id, doc_id, chunk_type, text, file_name, title, upload_date, metadata (object);
  Parents: no 'embedding', no 'parent_id'. Children: have 'embedding' (dense_vector) + 'parent_id' for KNN + join.
Index name: lease_documents


In [12]:
def _chunk_to_source(chunk: Dict[str, Any]) -> Dict[str, Any]:
    """Build an ES _source dict from a chunk, including optional parent_id and embedding."""
    src = {
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "chunk_type": chunk["chunk_type"],
        "chunk_index": chunk["chunk_index"],
        "text": chunk["text"],
        "file_name": chunk["file_name"],
        "title": chunk.get("title", ""),
        "upload_date": chunk.get("upload_date"),
        "metadata": chunk.get("metadata", {}),
    }
    if "parent_id" in chunk:
        src["parent_id"] = chunk["parent_id"]
    if "embedding" in chunk:
        src["embedding"] = chunk["embedding"]
    return src


def index_documents(documents: List[Dict[str, Any]], index_name: str) -> None:
    """Bulk-index parent and child chunks."""
    actions = []
    for doc in documents:
        for chunk in doc.get("parent_chunks", []) + doc.get("child_chunks", []):
            actions.append({"_index": index_name, "_id": chunk["chunk_id"], "_source": _chunk_to_source(chunk)})
    if actions:
        helpers.bulk(es, actions, raise_on_error=False)
        es.indices.refresh(index=index_name)
    n_parents = sum(len(d.get("parent_chunks", [])) for d in documents)
    n_children = sum(len(d.get("child_chunks", [])) for d in documents)
    print("Indexed %d chunks (%d parents, %d children) into '%s'." % (len(actions), n_parents, n_children, index_name))

if raw_documents:
    index_documents(raw_documents, ES_INDEX)

Indexed 543 chunks (94 parents, 449 children) into 'lease_documents'.


### Parent vs child: what is stored

| Field | **Parent** chunk | **Child** chunk |
|-------|------------------|-----------------|
| `chunk_id` | e.g. `parent_30e4d2430fc9ae36` | e.g. `child_5bc68125b4f6ae57` |
| `doc_id` | Same for all chunks of one document | Same |
| `chunk_type` | `"parent"` | `"child"` |
| `parent_id` | **Not stored** (N/A) | Parent’s `chunk_id` (for context join) |
| `text` | ~2000 chars (full section) | ~512 chars (sub-chunk for retrieval) |
| `embedding` | **Not stored** (no vector search on parents) | 768-dim vector (for KNN) |
| `file_name`, `title`, `upload_date` | Same as child | Same as parent |
| `metadata` | **Same object** on both (see below) | **Same object** (doc-level after enrichment) |

**Metadata object** (from `LeaseMetadata` schema) — **before enrichment** all values are `None`; **after running the enrichment pipeline** they are filled per document:

```python
# Example: after enrichment (one document)
metadata = {
    "owner_name": "John Doe",           # tenant / lessee
    "landlord": "Jane Smith",
    "signing_date": "2024-01-15",
    "expiry_date": "2025-01-14",
    "escalation_clause": "3% annual",
    "property_address": "123 Main St, City",
    "monthly_rent": "$2,500",
    "amendment_number": None,            # original lease
    "document_type": "lease"
}
```

For **parent_child** mode, the pipeline writes one **document-level** metadata to **all** chunks (parents + children) of that document, so every chunk is filterable by the same keys.

### Verify: metadata layer in ES

Quick sanity check — one parent and one child from the index (excluding large text/embedding fields).

In [13]:
# Example: one parent and one child (excludes text and embedding for readability)
import json
if not raw_documents:
    print("No raw_documents. Run the PDF ingestion and chunking cells (Sections 2-3) first.")
else:
    doc_id = raw_documents[0]["doc_id"]
    for label, chunk_type in [("PARENT", "parent"), ("CHILD", "child")]:
        try:
            r = es.search(
                index=ES_INDEX,
                body={
                    "query": {"bool": {"must": [{"term": {"doc_id": doc_id}}, {"term": {"chunk_type": chunk_type}}]}},
                    "size": 1,
                    "_source": {"excludes": ["text", "embedding"]},
                },
            )
        except Exception as e:
            print("ES search failed (is Elasticsearch running?):", e)
            break
        if r["hits"]["hits"]:
            print("%s example:" % label)
            print(json.dumps(r["hits"]["hits"][0]["_source"], indent=2))
            print()
        else:
            print("No %s chunks in index for doc_id=%s." % (chunk_type, doc_id))
            print("  -> Re-run the 'index_documents' cell (the one with index_documents(raw_documents, ES_INDEX)).")
            print("  -> Note: running 'Create Elasticsearch index' deletes the index; run 'index_documents' again after it.")

PARENT example:
{
  "chunk_id": "parent_30e4d2430fc9ae36",
  "doc_id": "doc_460c99c1193af42c",
  "chunk_type": "parent",
  "chunk_index": 0,
  "file_name": "california-standard-residential-lease-agreement-form.pdf",
  "title": "california standard residential lease agreement form",
  "upload_date": "2026-03-17",
  "metadata": {
    "document_type": null,
    "source_file": null
  }
}

CHILD example:
{
  "chunk_id": "child_5bc68125b4f6ae57",
  "doc_id": "doc_460c99c1193af42c",
  "chunk_type": "child",
  "chunk_index": 0,
  "file_name": "california-standard-residential-lease-agreement-form.pdf",
  "title": "california standard residential lease agreement form",
  "upload_date": "2026-03-17",
  "metadata": {
    "document_type": null,
    "source_file": null
  },
  "parent_id": "parent_30e4d2430fc9ae36"
}



In [14]:
if not raw_documents:
    print("No raw_documents. Run the PDF ingestion and chunking cells (Sections 2-3) first.")
else:
    doc_id = raw_documents[0]["doc_id"]
    exclude_fields = ["text", "embedding"]
    for chunk_type in ["parent", "child"]:
        try:
            r = es.search(
                index=ES_INDEX,
                body={
                    "query": {"bool": {"must": [{"term": {"doc_id": doc_id}}, {"term": {"chunk_type": chunk_type}}]}},
                    "size": 1,
                    "_source": {"excludes": exclude_fields},
                },
            )
        except Exception as e:
            print("ES search failed (is Elasticsearch running?):", e)
            break
        if r["hits"]["hits"]:
            src = r["hits"]["hits"][0]["_source"]
            print("=== %s chunk ===" % chunk_type)
            for k, v in src.items():
                print("  %s: %s" % (k, v))
            print("  has_embedding:", chunk_type == "child")
            print()
        else:
            print("No %s chunks in index." % chunk_type)
            print("  -> Re-run the 'index_documents' cell (index_documents(raw_documents, ES_INDEX)).")
            print("  -> Note: 'Create Elasticsearch index' wipes the index; run 'index_documents' again after it.")

=== parent chunk ===
  chunk_id: parent_30e4d2430fc9ae36
  doc_id: doc_460c99c1193af42c
  chunk_type: parent
  chunk_index: 0
  file_name: california-standard-residential-lease-agreement-form.pdf
  title: california standard residential lease agreement form
  upload_date: 2026-03-17
  metadata: {'document_type': None, 'source_file': None}
  has_embedding: False

=== child chunk ===
  chunk_id: child_5bc68125b4f6ae57
  doc_id: doc_460c99c1193af42c
  chunk_type: child
  chunk_index: 0
  file_name: california-standard-residential-lease-agreement-form.pdf
  title: california standard residential lease agreement form
  upload_date: 2026-03-17
  metadata: {'document_type': None, 'source_file': None}
  parent_id: parent_30e4d2430fc9ae36
  has_embedding: True



## 4. Fetch Chunks from Elasticsearch

Supports **flat** (all chunks) and **parent-child** (children only, for retrieval; they have embeddings). Parent text is fetched separately for extraction context. Chunks must have `text` and `embedding` for retrieval-augmented extraction.

In [15]:
def get_unique_doc_ids(es_client: Elasticsearch, index_name: str) -> List[str]:
    result = es_client.search(
        index=index_name,
        body={"size": 0, "aggs": {"doc_ids": {"terms": {"field": "doc_id", "size": 10000}}}},
    )
    return [b["key"] for b in result["aggregations"]["doc_ids"]["buckets"]]


def fetch_chunks_for_doc(
    es_client: Elasticsearch,
    index_name: str,
    doc_id: str,
    chunking_mode: str = "flat",
    include_embedding: bool = True,
) -> List[Dict[str, Any]]:
    """
    Fetch chunks for a document. For flat: all chunks. For parent_child: children only (for retrieval; they have embeddings).
    """
    if chunking_mode == "parent_child":
        query = {"bool": {"must": [{"term": {"doc_id": doc_id}}, {"term": {"chunk_type": "child"}}]}}
    else:
        query = {"term": {"doc_id": doc_id}}

    source = ["chunk_id", "doc_id", "chunk_index", "text", "file_name"]
    if chunking_mode == "parent_child":
        source.append("parent_id")
    if include_embedding:
        source.append("embedding")

    result = es_client.search(
        index=index_name,
        body={"query": query, "sort": [{"chunk_index": "asc"}], "size": 1000, "_source": source},
    )

    chunks = []
    for hit in result["hits"]["hits"]:
        src = hit["_source"]
        chunk = {
            "chunk_id": src.get("chunk_id", hit["_id"]),
            "doc_id": src.get("doc_id", doc_id),
            "chunk_index": src.get("chunk_index", 0),
            "text": src.get("text", ""),
            "file_name": src.get("file_name", ""),
        }
        if "parent_id" in src:
            chunk["parent_id"] = src["parent_id"]
        if "embedding" in src:
            chunk["embedding"] = src["embedding"]
        chunks.append(chunk)
    return chunks


def fetch_parent_chunks_for_doc(
    
    es_client: Elasticsearch,
    index_name: str,
    doc_id: str,
) -> List[Dict[str, Any]]:
    """Fetch parent chunks for a document (for parent_id -> text map in parent_child mode)."""
    result = es_client.search(
        index=index_name,
        body={
            "query": {"bool": {"must": [{"term": {"doc_id": doc_id}}, {"term": {"chunk_type": "parent"}}]}},
            "sort": [{"chunk_index": "asc"}],
            "size": 1000,
            "_source": ["chunk_id", "text"],
        },
    )
    return [{"chunk_id": h["_source"]["chunk_id"], "text": h["_source"].get("text", "")} for h in result["hits"]["hits"]]

## 5. In-Doc Retrieval (Per-Field) — ES-First

For each metadata field, query Elasticsearch directly using **native hybrid search** (KNN + BM25 + RRF) scoped to the document's chunks. No chunks or embeddings are pulled into Python — ES does all the heavy lifting. If `field_retrieval_queries` are not configured for a field, the query is auto-generated from the field's Pydantic schema description.

In [16]:
RRF_K = 60


def _encode_query(embedding_model: SentenceTransformer, query: str) -> np.ndarray:
    try:
        return embedding_model.encode(query, normalize_embeddings=True, prompt_name="query")
    except TypeError:
        return embedding_model.encode(query, normalize_embeddings=True)


def _parse_hits(hits, doc_id: str) -> List[Dict[str, Any]]:
    """Convert ES hits into chunk dicts."""
    chunks = []
    for hit in hits:
        src = hit["_source"]
        chunk = {
            "chunk_id": src.get("chunk_id", hit["_id"]),
            "doc_id": src.get("doc_id", doc_id),
            "chunk_index": src.get("chunk_index", 0),
            "text": src.get("text", ""),
            "file_name": src.get("file_name", ""),
        }
        if "parent_id" in src:
            chunk["parent_id"] = src["parent_id"]
        chunks.append(chunk)
    return chunks


def es_hybrid_search_chunks(
    es_client: Elasticsearch,
    index_name: str,
    doc_id: str,
    query_text: str,
    query_embedding: np.ndarray,
    top_k: int,
    rrf_k: int = 60,
    chunking_mode: str = "parent_child",
) -> List[Dict[str, Any]]:
    """
    Hybrid search (BM25 + KNN) with RRF fusion, compatible with ES 8.x.
    Runs two separate queries and fuses ranks in Python.
    """
    fetch_k = max(top_k * 2, 20)
    source_fields = ["chunk_id", "doc_id", "chunk_index", "text", "file_name", "parent_id"]
    doc_filter = [{"term": {"doc_id": doc_id}}]
    if chunking_mode == "parent_child":
        doc_filter.append({"term": {"chunk_type": "child"}})

    bm25_result = es_client.search(
        index=index_name,
        body={
            "query": {"bool": {"must": doc_filter + [{"match": {"text": {"query": query_text}}}]}},
            "size": fetch_k,
            "_source": source_fields,
        },
    )
    knn_result = es_client.search(
        index=index_name,
        body={
            "knn": {
                "field": "embedding",
                "query_vector": query_embedding.tolist(),
                "k": fetch_k,
                "num_candidates": min(100, fetch_k * 4),
                "filter": {"bool": {"must": doc_filter}},
            },
            "size": fetch_k,
            "_source": source_fields,
        },
    )

    bm25_chunks = _parse_hits(bm25_result["hits"]["hits"], doc_id)
    knn_chunks = _parse_hits(knn_result["hits"]["hits"], doc_id)

    chunk_by_id: Dict[str, Dict[str, Any]] = {}
    rrf_scores: Dict[str, float] = {}
    for ranked_list in [bm25_chunks, knn_chunks]:
        for rank_1, c in enumerate(ranked_list, start=1):
            cid = c["chunk_id"]
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (rrf_k + rank_1)
            chunk_by_id[cid] = c

    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
    return [chunk_by_id[cid] for cid in sorted_ids[:top_k]]


def _get_field_queries(schema: Type[BaseModel], field_name: str) -> tuple:
    """Build retrieval queries from the schema: (description, keywords, all_queries)."""
    field_info = schema.model_fields.get(field_name)
    desc = field_info.description if field_info and field_info.description else field_name.replace("_", " ")
    extra = (field_info.json_schema_extra or {}) if field_info else {}
    keywords = extra.get("keywords", [])
    return desc, keywords, [desc] + keywords


def retrieve_chunks_for_field(
    es_client: Elasticsearch,
    index_name: str,
    doc_id: str,
    field_name: str,
    embedding_model: SentenceTransformer,
    schema: Type[BaseModel],
    top_k: int = 5,
    chunking_mode: str = "parent_child",
) -> List[Dict[str, Any]]:
    """
    Retrieve top-k chunks using hybrid search (KNN + BM25 + RRF).
    Queries are built from the schema: description (primary) + keywords (supplementary).
    """
    _, _, queries = _get_field_queries(schema, field_name)

    rrf_scores: Dict[str, float] = {}
    chunk_by_id: Dict[str, Dict[str, Any]] = {}
    for query in queries:
        query_emb = _encode_query(embedding_model, query)
        ranked = es_hybrid_search_chunks(
            es_client, index_name, doc_id, query, query_emb,
            top_k=max(top_k * 2, 20), rrf_k=RRF_K, chunking_mode=chunking_mode,
        )
        for rank_1based, c in enumerate(ranked, start=1):
            cid = c["chunk_id"]
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (RRF_K + rank_1based)
            chunk_by_id[cid] = c

    sorted_cids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
    return [chunk_by_id[cid] for cid in sorted_cids[:top_k]]


def retrieve_all_fields_chunks(
    es_client: Elasticsearch,
    index_name: str,
    doc_id: str,
    config: "EnrichmentConfig",
    embedding_model: SentenceTransformer,
) -> List[Dict[str, Any]]:
    """
    For each metadata field, run ES hybrid search to get top-k chunks.
    Deduplicate across all fields and return a single list of unique chunks,
    ordered by how many fields retrieved them (most relevant first).
    """
    schema = config.metadata_schema
    field_names = list(schema.model_fields.keys())
    chunk_by_id: Dict[str, Dict[str, Any]] = {}
    chunk_hit_count: Dict[str, int] = {}

    for field_name in tqdm(field_names, desc="Retrieving chunks per field"):
        retrieved = retrieve_chunks_for_field(
            es_client, index_name, doc_id, field_name,
            embedding_model, schema, config.top_k_per_field, config.chunking_mode,
        )
        for c in retrieved:
            cid = c["chunk_id"]
            chunk_by_id[cid] = c
            chunk_hit_count[cid] = chunk_hit_count.get(cid, 0) + 1

    sorted_cids = sorted(chunk_hit_count, key=chunk_hit_count.get, reverse=True)
    return [chunk_by_id[cid] for cid in sorted_cids]

## 5b. Inspect Retrieved Chunks per Field (Before LLM Context)

For a given document, show which chunks were retrieved for **each metadata field** (the same retrieval that feeds the LLM). This is the per-field context before it is merged and sent to `extract_all_fields`.

In [18]:
def _fetch_parent_texts_inline(
    es_client: Elasticsearch, index_name: str, parent_ids: List[str],
) -> Dict[str, str]:
    """Fetch parent chunk texts by their IDs."""
    if not parent_ids:
        return {}
    result = es_client.search(
        index=index_name,
        body={
            "query": {"ids": {"values": parent_ids}},
            "size": len(parent_ids),
            "_source": ["chunk_id", "text"],
        },
    )
    return {h["_source"]["chunk_id"]: h["_source"].get("text", "") for h in result["hits"]["hits"]}


def _build_prompt(context: str, schema: Type[BaseModel]) -> str:
    """Build the exact prompt that extract_all_fields sends to the LLM."""
    field_descriptions = "\n".join(
        f"- {name}: {info.description or name}"
        for name, info in schema.model_fields.items()
    )
    return f"""Extract the following metadata fields from the document text.
Return null for any field whose value cannot be determined.

Fields to extract:
{field_descriptions}

--- DOCUMENT TEXT ---
{context[:30000]}
--- END ---

Return a JSON object with all the fields above."""


def inspect_retrieval_and_prompt(
    es_client: Elasticsearch,
    index_name: str,
    doc_id: str,
    config: "EnrichmentConfig",
    embedding_model: SentenceTransformer,
    child_text_max_len: int = 150,
    parent_text_max_len: int = 400,
) -> None:
    """
    Full inspection for one document — mirrors enrich_document_retrieval_augmented exactly:
      1. Per-field retrieval (child chunks + queries used)
      2. Deduplicate → resolve to parent chunks (the actual LLM context)
      3. Print the assembled parent context
      4. Print the full prompt that would be sent to the LLM
    """
    schema = config.metadata_schema
    field_names = list(schema.model_fields.keys())

    # ── Step 1: per-field retrieval ──
    all_child_chunks: Dict[str, Dict[str, Any]] = {}
    child_hit_count: Dict[str, int] = {}

    print(f"Document: {doc_id}")
    print("=" * 80)
    print("\n── STEP 1: Per-field retrieval (child chunks) ──")
    for field_name in field_names:
        desc, keywords, all_queries = _get_field_queries(schema, field_name)
        chunks = retrieve_chunks_for_field(
            es_client, index_name, doc_id, field_name,
            embedding_model, schema, config.top_k_per_field, config.chunking_mode,
        )
        print(f"\n  Field: {field_name}")
        print(f"  Queries ({len(all_queries)}): [{desc}] + {len(keywords)} keywords")
        print(f"  Children retrieved: {len(chunks)}")
        for r, c in enumerate(chunks, start=1):
            cid = c.get("chunk_id", "")
            pid = c.get("parent_id", "—")
            text = (c.get("text") or "")[:child_text_max_len]
            if len(c.get("text") or "") > child_text_max_len:
                text += "..."
            print(f"    [{r}] {cid}  →  parent: {pid}")
            print(f"        {repr(text)}")

        for c in chunks:
            cid = c["chunk_id"]
            all_child_chunks[cid] = c
            child_hit_count[cid] = child_hit_count.get(cid, 0) + 1

    # ── Step 2: deduplicate and resolve parents ──
    sorted_cids = sorted(child_hit_count, key=child_hit_count.get, reverse=True)
    deduped_chunks = [all_child_chunks[cid] for cid in sorted_cids]

    print(f"\n\n── STEP 2: Deduplicated child chunks → parent resolution ──")
    print(f"  Unique child chunks across all fields: {len(deduped_chunks)}")

    if config.chunking_mode == "parent_child":
        parent_ids = list({c["parent_id"] for c in deduped_chunks if c.get("parent_id")})
        parent_texts = _fetch_parent_texts_inline(es_client, index_name, parent_ids)
        print(f"  Unique parent chunks resolved: {len(parent_texts)}")

        print(f"\n  Child → Parent mapping:")
        child_to_parent: Dict[str, str] = {}
        for c in deduped_chunks:
            pid = c.get("parent_id", "—")
            child_to_parent[c["chunk_id"]] = pid
        for cid, pid in child_to_parent.items():
            hit_count = child_hit_count[cid]
            print(f"    {cid} → {pid}  (retrieved by {hit_count} field{'s' if hit_count > 1 else ''})")

        context_parts = [parent_texts[pid] for pid in parent_ids if pid in parent_texts]
        context = "\n\n".join(context_parts) if context_parts else "\n\n".join(c["text"] for c in deduped_chunks)
    else:
        context = "\n\n".join(c["text"] for c in deduped_chunks)
        parent_ids = []
        parent_texts = {}

    # ── Step 3: show assembled parent context ──
    print(f"\n\n── STEP 3: Assembled context (parent chunks) ──")
    if config.chunking_mode == "parent_child" and parent_texts:
        for i, pid in enumerate(parent_ids, start=1):
            ptext = parent_texts.get(pid, "")
            truncated = ptext[:parent_text_max_len]
            if len(ptext) > parent_text_max_len:
                truncated += "..."
            print(f"\n  Parent [{i}] {pid} ({len(ptext)} chars)")
            print(f"  {repr(truncated)}")
    else:
        print(f"  (flat mode — using child chunk texts directly, {len(deduped_chunks)} chunks)")

    print(f"\n  Total context length: {len(context)} chars")

    # ── Step 4: show full prompt ──
    prompt = _build_prompt(context, schema)
    print(f"\n\n── STEP 4: Full LLM prompt ({len(prompt)} chars) ──")
    print("┌" + "─" * 78 + "┐")
    for line in prompt.split("\n"):
        print(f"│ {line[:76]:<76} │")
    print("└" + "─" * 78 + "┘")


# Run inspection on the first document
_result = es.search(
    index=ES_INDEX,
    body={"size": 0, "aggs": {"doc_ids": {"terms": {"field": "doc_id", "size": 10000}}}},
)
_doc_ids = [b["key"] for b in _result["aggregations"]["doc_ids"]["buckets"]]
if _doc_ids:
    inspect_retrieval_and_prompt(es, ES_INDEX, _doc_ids[0], config, embedding_model)
else:
    print("No documents in index. Run ingestion and indexing first.")

Document: doc_460c99c1193af42c

── STEP 1: Per-field retrieval (child chunks) ──

  Field: owner_name
  Queries (9): [Name of the tenant, lessee, or occupant who rents the property] + 8 keywords
  Children retrieved: 5
    [1] child_502f5962c1bbf322  →  parent: parent_34d2abf8a83cb64a
        "  term  of  this  California  Lease  Agreement  Tenant abandons  the  Premises  or  any  part  thereof,  Landlord  may,  at  Landlord's  option,  obta..."
    [2] child_2508d8c9bb70abf4  →  parent: parent_c7aada336b8f13be
        "any other person, other than Tenant's immediate family or transient relatives and friends who are guests of Tenant, to use or occupy the Premises with..."
    [3] child_7b9e562f92367c94  →  parent: parent_975e9606033f15bc
        'OVEMENTS .   Tenant  shall  make  no  alterations  to  the  buildings  or improvements on the Premises or construct any building or make any other imp...'
    [4] child_d90d596b882297f2  →  parent: parent_c7aada336b8f13be
        'immediate fa

## 6. LLM Extraction (Batch — All Fields at Once)

Extract **all** metadata fields in a single LLM call using the full Pydantic schema as Gemini's structured output constraint. The combined context from all per-field retrievals is passed once, and the LLM returns the complete metadata dict.

In [ ]:
def extract_all_fields(
    text: str,
    schema: Type[BaseModel],
    model: str = GEMINI_MODEL,
) -> Dict[str, Any]:
    """
    Extract ALL metadata fields in one LLM call using the full Pydantic schema.
    Returns a dict with one key per field (value or None).
    """
    field_descriptions = "\n".join(
        f"- {name}: {info.description or name}"
        for name, info in schema.model_fields.items()
    )
    prompt = f"""Extract the following metadata fields from the document text.
Return null for any field whose value cannot be determined.

Fields to extract:
{field_descriptions}

--- DOCUMENT TEXT ---
{text[:30000]}
--- END ---

Return a JSON object with all the fields above."""

    try:
        # response = gemini_client.models.generate_content(
        #     model=model,
        #     contents=prompt,
        #     config={
        #         "response_mime_type": "application/json",
        #         "response_schema": schema.model_json_schema(),
        #     },
        # )
        # parsed = schema.model_validate_json(response.text)
        return parsed.model_dump()
    except Exception as e:
        print(f"  [WARN] Batch extraction failed: {e}")
        return {name: None for name in schema.model_fields}

## 7. Retrieval-Augmented Enricher (ES-First, Batch)

Per-field ES hybrid search (KNN + BM25 + RRF) → deduplicate chunks across all fields → fetch parent texts on-demand → **one LLM call** with the full schema to extract all fields at once. No chunks are pre-fetched; N fields = 1 LLM call per document.

In [ ]:
def _fetch_parent_texts(
    es_client: Elasticsearch, index_name: str, parent_ids: List[str],
) -> Dict[str, str]:
    """Fetch parent chunk texts for a specific set of parent_ids (on-demand, not all parents)."""
    if not parent_ids:
        return {}
    result = es_client.search(
        index=index_name,
        body={
            "query": {"ids": {"values": parent_ids}},
            "size": len(parent_ids),
            "_source": ["chunk_id", "text"],
        },
    )
    return {h["_source"]["chunk_id"]: h["_source"].get("text", "") for h in result["hits"]["hits"]}


def enrich_document_retrieval_augmented(
    es_client: Elasticsearch,
    index_name: str,
    doc_id: str,
    config: EnrichmentConfig,
    embedding_model: SentenceTransformer,
) -> Dict[str, Any]:
    """
    ES-first batch enrichment:
      1. For each field → ES hybrid search → top-k chunks
      2. Deduplicate chunks across all fields
      3. If parent_child → fetch parent texts for unique parent_ids only
      4. ONE LLM call with full schema → extract all fields at once
    Returns doc_metadata dict. No chunks are pre-fetched into Python.
    """
    all_chunks = retrieve_all_fields_chunks(
        es_client, index_name, doc_id, config, embedding_model,
    )
    if not all_chunks:
        return {f: None for f in config.metadata_schema.model_fields}

    if config.chunking_mode == "parent_child":
        parent_ids = list({c["parent_id"] for c in all_chunks if c.get("parent_id")})
        parent_texts = _fetch_parent_texts(es_client, index_name, parent_ids)
        context_parts = [parent_texts[pid] for pid in parent_ids if pid in parent_texts]
        context = "\n\n".join(context_parts) if context_parts else "\n\n".join(c["text"] for c in all_chunks)
    else:
        context = "\n\n".join(c["text"] for c in all_chunks)

    if not context.strip():
        return {f: None for f in config.metadata_schema.model_fields}

    n_chunks = len(all_chunks)
    n_fields = len(config.metadata_schema.model_fields)
    print(f"  Batch extract: {n_chunks} unique chunks → 1 LLM call for {n_fields} fields")

    return extract_all_fields(context, config.metadata_schema)

## 8. Section-Based Fallback + Merge

When retrieval-augmented is not possible (e.g. no embeddings) or as fallback: group chunks into sections, extract per section, merge results, assign to chunks. `merge_extractions` consolidates multiple section-level dicts into one doc-level dict.

In [ ]:
def merge_extractions(
    extractions: List[Dict[str, Any]],
    strategy: str = "first_non_null",
) -> Dict[str, Any]:
    """Merge multiple section-level extractions into one doc-level dict."""
    if not extractions:
        return {}
    all_fields = {k for ext in extractions for k in ext}
    merged = {}
    for field in all_fields:
        values = [ext[field] for ext in extractions if ext.get(field) is not None]
        if not values:
            merged[field] = None
        elif strategy == "last_non_null":
            merged[field] = values[-1]
        elif strategy == "most_common":
            merged[field] = max(set(values), key=values.count)
        else:
            merged[field] = values[0]
    return merged


def compute_section_size(config: EnrichmentConfig, chunk_size_chars: int = 512) -> int:
    if config.chunks_per_section is not None:
        return config.chunks_per_section
    max_chars = int(config.context_limit_tokens * config.chars_per_token * config.safety_margin)
    return max(1, max_chars // chunk_size_chars)


def enrich_document_section_based(
    chunks: List[Dict[str, Any]],
    config: EnrichmentConfig,
) -> tuple:
    """Section-based extraction: group chunks, extract per section, merge. Returns (chunks, doc_metadata)."""
    section_size = compute_section_size(config, len(chunks[0].get("text", "")) if chunks else 512)
    sections = [chunks[i:i + section_size] for i in range(0, len(chunks), section_size)]
    schema_desc = "\n".join(f"  - {n}: {f.description or n}" for n, f in config.metadata_schema.model_fields.items())
    extractions = []

    for section_chunks in tqdm(sections, desc="Extracting sections"):
        section_text = "\n\n".join(c.get("text", "") for c in section_chunks)[:15000]
        prompt = f"""Extract the requested metadata fields from this document section.
If a field cannot be determined, return null.

Schema:\n{schema_desc}

--- DOCUMENT SECTION ---
{section_text}
--- END ---

Return ONLY a JSON object matching the schema."""
        try:
            meta = config.metadata_schema.model_validate_json(
                gemini_client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=prompt,
                    config={"response_mime_type": "application/json", "response_schema": config.metadata_schema},
                ).text
            )
            extractions.append(meta.model_dump())
        except Exception as e:
            extractions.append({f: None for f in config.metadata_schema.model_fields})
            print(f"  [WARN] Section extraction failed: {e}")

    merged = merge_extractions(extractions, config.merge_strategy)
    for chunk in chunks:
        chunk["metadata"] = dict(merged)
    return chunks, merged

## 9. Main Enrichment Flow and Re-Index

ES-first pipeline: for each document we (1) **Query ES per field** with hybrid search (KNN + BM25 + RRF), (2) **Extract** values with the LLM from retrieved context, (3) **Collect** document-level metadata, (4) **Bulk-update** all chunk_ids for that `doc_id` with the enriched metadata. No embeddings or full chunk lists are loaded into Python.

In [ ]:
def _doc_already_enriched(es_client: Elasticsearch, index_name: str, doc_id: str) -> bool:
    r = es_client.search(
        index=index_name,
        body={"query": {"term": {"doc_id": doc_id}}, "size": 1, "_source": ["metadata"]},
    )
    if not r["hits"]["hits"]:
        return False
    meta = r["hits"]["hits"][0]["_source"].get("metadata", {})
    return any(v is not None for v in meta.values())


def _get_all_chunk_ids(es_client: Elasticsearch, index_name: str, doc_id: str) -> List[str]:
    """Lightweight query: get all chunk_ids for a doc (parents + children), no text/embeddings."""
    result = es_client.search(
        index=index_name,
        body={"query": {"term": {"doc_id": doc_id}}, "size": 10000, "_source": ["chunk_id"]},
    )
    return [h["_source"]["chunk_id"] for h in result["hits"]["hits"]]


def _build_update_actions(
    index_name: str, chunk_ids: List[str], doc_metadata: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """Build ES bulk-update actions to write the same doc_metadata to all chunks of a document."""
    return [
        {"_op_type": "update", "_index": index_name, "_id": cid, "doc": {"metadata": doc_metadata}}
        for cid in chunk_ids
    ]


def run_enrichment_pipeline(
    es_client: Elasticsearch,
    index_name: str,
    config: EnrichmentConfig,
    embedding_model: SentenceTransformer,
    overwrite: bool = False,
):
    """Enrich all documents: per-field ES hybrid search → LLM extract → bulk-update ES."""
    doc_ids = get_unique_doc_ids(es_client, index_name)
    print(f"Found {len(doc_ids)} documents in '{index_name}'")

    for doc_id in tqdm(doc_ids, desc="Enriching documents"):
        if not overwrite and _doc_already_enriched(es_client, index_name, doc_id):
            continue

        if config.extraction_strategy == "retrieval_augmented":
            doc_metadata = enrich_document_retrieval_augmented(
                es_client, index_name, doc_id, config, embedding_model,
            )
        else:
            chunks = fetch_chunks_for_doc(es_client, index_name, doc_id, config.chunking_mode, include_embedding=False)
            if not chunks:
                print(f"  [WARN] No chunks for doc_id={doc_id}")
                continue
            _, doc_metadata = enrich_document_section_based(chunks, config)

        preview = {k: (v[:50] + "..." if isinstance(v, str) and len(v) > 50 else v) for k, v in doc_metadata.items()}
        print(f"  doc_id={doc_id}: {preview}")

        chunk_ids = _get_all_chunk_ids(es_client, index_name, doc_id)
        actions = _build_update_actions(index_name, chunk_ids, doc_metadata)
        helpers.bulk(es_client, actions, raise_on_error=False)

    es_client.indices.refresh(index=index_name)
    print("Enrichment complete.")

In [ ]:
# Run the enrichment pipeline
run_enrichment_pipeline(es, ES_INDEX, config, embedding_model, overwrite=True)

## 9b. Enriched Content (Document-Wise)

Display the extracted metadata for each document in the index. Each document's enriched fields are shown in a readable format.

In [ ]:
def show_enriched_content_document_wise(
    es_client: Elasticsearch,
    index_name: str,
    max_docs: int | None = None,
) -> None:
    """
    Fetch and print enriched metadata for each document.
    Uses one chunk per doc (metadata is identical across all chunks of a doc).
    """
    doc_ids = get_unique_doc_ids(es_client, index_name)
    if not doc_ids:
        print("No documents in index.")
        return
    if max_docs is not None:
        doc_ids = doc_ids[:max_docs]

    for i, doc_id in enumerate(doc_ids, start=1):
        result = es_client.search(
            index=index_name,
            body={
                "query": {"term": {"doc_id": doc_id}},
                "size": 1,
                "_source": ["file_name", "metadata"],
            },
        )
        hits = result.get("hits", {}).get("hits", [])
        if not hits:
            print(f"  [{i}] doc_id={doc_id}: no chunks")
            continue
        src = hits[0]["_source"]
        file_name = src.get("file_name", "—")
        meta = src.get("metadata", {})

        print(f"\n{'='*60}")
        print(f"  Document {i}: {doc_id}")
        print(f"  File: {file_name}")
        print(f"{'='*60}")
        for key, value in meta.items():
            if value is None:
                display_val = "(null)"
            elif isinstance(value, str) and len(value) > 80:
                display_val = value[:80] + "..."
            else:
                display_val = value
            print(f"  {key}: {display_val}")
    print(f"\nTotal: {len(doc_ids)} document(s)")


# Show enriched content for all documents (set max_docs=5 to limit)
show_enriched_content_document_wise(es, ES_INDEX, max_docs=None)

## 10. Validation: Per-Chunk Metadata (Value vs Null)

Verify that retrieved chunks have values and non-retrieved chunks have null.

In [ ]:
# Sample a few chunks and show metadata (value vs null)
doc_ids = get_unique_doc_ids(es, ES_INDEX)
if not doc_ids:
    print("No documents in index.")
else:
    sample = es.search(
        index=ES_INDEX,
        body={"query": {"term": {"doc_id": doc_ids[0]}}, "size": 8, "_source": ["chunk_id", "text", "metadata"]},
    )
    print("Per-chunk metadata (value vs null):\n")
    for hit in sample.get("hits", {}).get("hits", []):
        src = hit["_source"]
        meta = src.get("metadata", {})
        non_null = {k: v for k, v in meta.items() if v is not None}
        null_count = sum(1 for v in meta.values() if v is None)
        print(f"  {src['chunk_id'][:20]}... | non-null: {len(non_null)}, null: {null_count}")
        if non_null:
            print(f"    {non_null}")

---

## Summary: v2 Architecture

| Feature | Implementation |
|---------|----------------|
| **Chunking** | Flat or parent-child via `config.chunking_mode` |
| **Extraction** | Retrieval-augmented (primary) or section-based (fallback) |
| **Per-chunk metadata** | Retrieved chunks: value; others: explicit null |
| **Context window** | Never exceeded — only retrieved chunks or sections sent to LLM |
| **Generic** | Schema-driven: description + keywords per field via `json_schema_extra` |

Use **Notebook 2** (`02_metadata_retrieval_pipeline.ipynb`) for retrieval. Optional: filter by `metadata.field != null` for granular retrieval.

## Appendix: Index Inspection and Cleanup

Utility cells to inspect what's in the Elasticsearch index and how to delete it. Separate cells for ES 8.x and ES 9.x syntax where they differ.

In [ ]:
# ── Inspect: list all indices and their doc counts (ES 8 & 9) ──
indices = es.cat.indices(format="json")
print(f"{'Index':<35} {'Docs':>8}  {'Size':>10}  {'Status'}")
print("-" * 75)
for idx in sorted(indices, key=lambda x: x["index"]):
    print(f"{idx['index']:<35} {idx.get('docs.count', '?'):>8}  {idx.get('store.size', '?'):>10}  {idx.get('health', '?')}")

In [ ]:
# ── Inspect: what's inside a specific index (ES 8 & 9) ──
INDEX_TO_INSPECT = ES_INDEX  # change to any index name

# 1. Index mapping (field types)
mapping = es.indices.get_mapping(index=INDEX_TO_INSPECT)
print(f"=== Mapping for '{INDEX_TO_INSPECT}' ===")
for field, props in mapping[INDEX_TO_INSPECT]["mappings"].get("properties", {}).items():
    ftype = props.get("type", "object")
    if ftype == "object" and "properties" in props:
        subfields = list(props["properties"].keys())
        print(f"  {field}: {ftype} → {subfields}")
    else:
        extra = f" (dims={props['dims']})" if "dims" in props else ""
        print(f"  {field}: {ftype}{extra}")

# 2. Doc count and unique doc_ids
count = es.count(index=INDEX_TO_INSPECT)["count"]
agg = es.search(
    index=INDEX_TO_INSPECT,
    body={"size": 0, "aggs": {"doc_ids": {"terms": {"field": "doc_id", "size": 10000}}}},
)
buckets = agg["aggregations"]["doc_ids"]["buckets"]
print(f"\n=== Stats for '{INDEX_TO_INSPECT}' ===")
print(f"  Total chunks: {count}")
print(f"  Unique documents: {len(buckets)}")
for b in buckets:
    print(f"    doc_id={b['key']}  chunks={b['doc_count']}")

# 3. Sample a few chunks
sample = es.search(index=INDEX_TO_INSPECT, body={"size": 3, "_source": {"excludes": ["embedding"]}})
print(f"\n=== Sample chunks (first 3) ===")
for hit in sample["hits"]["hits"]:
    src = hit["_source"]
    text_preview = (src.get("text") or "")[:120]
    print(f"  {src.get('chunk_id', hit['_id'])} | type={src.get('chunk_type', '?')} | doc_id={src.get('doc_id', '?')}")
    print(f"    text: {text_preview}{'...' if len(src.get('text','')) > 120 else ''}")
    print(f"    metadata: {src.get('metadata', {})}")

In [ ]:
# ── Delete an index: ES 8.x ──
# Works on Elasticsearch 8.x (uses the same API as ES 7.x)
# WARNING: This permanently deletes the index and all its data!

INDEX_TO_DELETE = ES_INDEX  # change to the index you want to delete

# Uncomment below to execute:
# if es.indices.exists(index=INDEX_TO_DELETE):
#     es.indices.delete(index=INDEX_TO_DELETE)
#     print(f"Index '{INDEX_TO_DELETE}' deleted.")
# else:
#     print(f"Index '{INDEX_TO_DELETE}' does not exist.")

print(f"To delete index '{INDEX_TO_DELETE}', uncomment the lines above and re-run.")

In [ ]:
# ── Delete an index: ES 9.x ──
# ES 9.x uses the same indices.delete() API but requires elasticsearch-py >= 9.x client.
# If using the ES 9.x Python client, the call is identical but with updated client headers.
# WARNING: This permanently deletes the index and all its data!

INDEX_TO_DELETE = ES_INDEX  # change to the index you want to delete

# For ES 9.x Python client (elasticsearch>=9.0.0):
# from elasticsearch import Elasticsearch
# es9 = Elasticsearch("http://localhost:9200", basic_auth=("elastic", "changeme"))
#
# if es9.indices.exists(index=INDEX_TO_DELETE):
#     es9.indices.delete(index=INDEX_TO_DELETE)
#     print(f"Index '{INDEX_TO_DELETE}' deleted (ES 9.x).")
# else:
#     print(f"Index '{INDEX_TO_DELETE}' does not exist.")
#
# ES 9.x also supports delete-by-query to remove specific documents without dropping the index:
# es9.delete_by_query(
#     index=INDEX_TO_DELETE,
#     body={"query": {"term": {"doc_id": "doc_id_to_remove"}}},
# )

print(f"To delete index '{INDEX_TO_DELETE}' on ES 9.x, uncomment the lines above and re-run.")